# 09 LightGBM Rolling-Origin Expanding-Window Evaluation

This notebook applies the validation-selected LightGBM configurations from notebook 08 to the reserved out-of-sample period using an expanding-window, forecast-origin-aware rolling-origin design. For each forecast origin, training uses rows with target dates strictly before the origin, and predictions are produced for the configured horizons at that origin.

The notebook reads the ML panel, feature registry and tuned LightGBM parameter file, evaluates M1-M4 on the original sales scale, and writes prediction, metric, runtime and validation outputs under `outputs/lightgbm_rolling_origin/`. Hyperparameters are not retuned here, and no raw or processed input data are modified.

M3 uses realised target-day point weather and is retained only as a non-operational perfect-information benchmark. M1, M2 and M4 follow the registry-defined operational feature restrictions.


## Configuration

The configuration defines the evaluation window, origin schedule, horizons, feature sets, output suffix and run size. Smoke mode limits the number of origins and horizons for validation; full mode evaluates all configured step-5 origins and horizons h=0..10. Distinct suffixes prevent smoke and full outputs from overwriting each other.


In [ ]:
RUN_MODE = "full"  # change to "full" for the thesis-level rolling-origin run.
assert RUN_MODE in {"smoke", "full"}, f"Unknown RUN_MODE: {RUN_MODE!r}"

# Step-5 origins rotate weekdays while keeping the run below daily-origin cost.
EVAL_START_DATE = "2024-08-01"
EVAL_END_DATE = "2025-07-31"
ORIGIN_SCHEDULE_MODE = "step"
ALLOWED_ORIGIN_SCHEDULE_MODES = {"step", "daily", "balanced_weekday"}
assert ORIGIN_SCHEDULE_MODE in ALLOWED_ORIGIN_SCHEDULE_MODES, (
    f"Unknown ORIGIN_SCHEDULE_MODE: {ORIGIN_SCHEDULE_MODE!r}"
)
ORIGIN_STEP_DAYS = 5

HORIZONS_FULL = list(range(0, 11))
HORIZONS_SMOKE = [0, 1, 2, 3, 7, 10]
FEATURE_SETS = ["M1", "M2", "M3", "M4"]
MODEL_FAMILY = "lightgbm_tuned"
RANDOM_STATE = 42
N_JOBS = -1
EST_MINUTES_PER_FIT = 0.56
REQUIRE_PREFLIGHT_PASS_FOR_FULL = True
ALLOW_NON_STEP5_FULL = False

if RUN_MODE == "smoke":
    HORIZONS_TO_RUN = HORIZONS_SMOKE
    MAX_ORIGINS = 12
else:
    HORIZONS_TO_RUN = HORIZONS_FULL
    MAX_ORIGINS = None

SCHEDULE_TAG = (
    f"{RUN_MODE}_step{ORIGIN_STEP_DAYS}"
    if ORIGIN_SCHEDULE_MODE == "step"
    else f"{RUN_MODE}_{ORIGIN_SCHEDULE_MODE}"
)
FILE_SUFFIX = f"_{SCHEDULE_TAG}"
CLIP_NEGATIVE_PREDICTIONS = True


## Environment, paths and helpers

This cell resolves project-relative paths, checks that upstream artifacts exist, and defines the rolling-origin output file scheme.


In [ ]:
import gc
import json
import time
import warnings
from html import escape as _html_escape
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Start Jupyter from inside the project folder."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_ML_PANEL_DIR = OUTPUT_DIR / "ml_panel"
OUTPUT_TUNING_DIR   = OUTPUT_DIR / "lightgbm_tuning"
OUTPUT_ROLLING_DIR  = OUTPUT_DIR / "lightgbm_rolling_origin"
FIG_DIR             = OUTPUT_ROLLING_DIR / f"figures{FILE_SUFFIX}"
PREFLIGHT_AUDIT_DIR = OUTPUT_ROLLING_DIR / "preflight_audit"

ML_PANEL_PATH = PROCESSED_DIR / "ml_forecast_panel_full.parquet"
FEATURE_REGISTRY_PATH = OUTPUT_ML_PANEL_DIR / "ml_panel_feature_registry.csv"
BEST_PARAMS_PATH = OUTPUT_TUNING_DIR / "lightgbm_optuna_best_params_full.json"
VAL_METRICS_PATH = OUTPUT_TUNING_DIR / "lightgbm_optuna_validation_metrics_full.csv"
TRIALS_PATH = OUTPUT_TUNING_DIR / "lightgbm_optuna_trials_full.csv"
RUNTIME_PATH_08 = OUTPUT_TUNING_DIR / "lightgbm_optuna_runtime_summary_full.csv"

OUTPUT_ROLLING_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT).as_posix()
    except ValueError:
        return str(path)


def require_file(path, description):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {description}: {project_relative(path)}. "
            "Rerun the producing notebook before running this notebook."
        )


# Require upstream artifacts before defining outputs.
for path, desc in [
    (ML_PANEL_PATH,         "ML forecast panel (from notebook 06)"),
    (FEATURE_REGISTRY_PATH, "feature registry (from notebook 06)"),
    (BEST_PARAMS_PATH,      "tuned LightGBM parameters (from notebook 08, full mode)"),
    (VAL_METRICS_PATH,      "08 tuning validation metrics"),
    (TRIALS_PATH,           "08 tuning trial log"),
    (RUNTIME_PATH_08,       "08 tuning runtime summary"),
]:
    require_file(path, desc)


# Use FILE_SUFFIX to isolate run modes and origin schedules.
def out_path(stem: str, ext: str) -> Path:
    return OUTPUT_ROLLING_DIR / f"lightgbm_rolling_origin_{stem}{FILE_SUFFIX}.{ext}"


PREDICTIONS_PATH         = out_path("predictions",                    "parquet")
METRICS_PATH             = out_path("metrics",                        "csv")
GAIN_SUMMARY_PATH        = out_path("gain_summary",                   "csv")
WEATHER_VALUE_DECOMP_PATH = out_path("weather_value_decomposition",   "csv")
M4_VS_M2_DIAG_PATH       = out_path("m4_vs_m2_diagnostic",            "csv")
HORIZON_TIER_PATH        = out_path("horizon_tier_summary",           "csv")
RUNTIME_SUMMARY_PATH     = out_path("runtime_summary",                "csv")
VALIDATION_CHECKS_PATH   = out_path("validation_checks",              "csv")
FEATURE_SET_SUMMARY_PATH = out_path("feature_set_summary",            "csv")
USED_PARAMS_JSON_PATH    = out_path("used_params",                    "json")
USED_PARAMS_CSV_PATH     = out_path("used_params",                    "csv")
METRICS_HTML_PATH        = out_path("metrics",                        "html")
GAIN_HTML_PATH           = out_path("gain_summary",                   "html")
REPORT_MD_PATH           = out_path("report",                         "md")
REPORT_HTML_PATH         = out_path("report",                         "html")

print(f"Project root:  {PROJECT_ROOT}")
print(f"ML panel:      {project_relative(ML_PANEL_PATH)}")
print(f"Registry:      {project_relative(FEATURE_REGISTRY_PATH)}")
print(f"Best params:   {project_relative(BEST_PARAMS_PATH)}")
print(f"Outputs:       {project_relative(OUTPUT_ROLLING_DIR)}")
print()
print(f"RUN_MODE:      {RUN_MODE}")
print(f"HORIZONS:      {HORIZONS_TO_RUN}")
print(f"FEATURE_SETS:  {FEATURE_SETS}")
print(f"Eval window:   {EVAL_START_DATE} .. {EVAL_END_DATE}")
print(f"Origin mode:   {ORIGIN_SCHEDULE_MODE}")
print(f"Step (days):   {ORIGIN_STEP_DAYS if ORIGIN_SCHEDULE_MODE == 'step' else 'n/a'}")
print(f"Max origins:   {MAX_ORIGINS if MAX_ORIGINS is not None else 'all in schedule'}")
print(f"Output suffix: {FILE_SUFFIX}")


## Imports and package availability

This cell imports modelling dependencies and records whether optional plotting support is available.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from lightgbm import LGBMRegressor  # noqa: F401
except Exception as exc:  # pragma: no cover
    raise ImportError(
        f"LightGBM not available: {type(exc).__name__}: {exc}. "
        "Install lightgbm in the csvi_env conda environment."
    )

# Plots are written only when matplotlib imports successfully.
try:
    import matplotlib.pyplot as plt
    _MPL_OK = True
except Exception:
    _MPL_OK = False

print(f"lightgbm: OK; matplotlib: {_MPL_OK}")


## Feature registry loading

The feature registry defines feature-set membership and leakage flags for M1-M4. The store and category identifiers are reintroduced only as intended categorical model inputs.


In [ ]:
feature_registry = pd.read_csv(FEATURE_REGISTRY_PATH)


def _to_bool(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


for col in [
    "allowed_m1_baseline",
    "allowed_m2_synthetic_point_weather",
    "allowed_m3_perfect_information",
    "allowed_m4_synthetic_engineered_weather",
    "leakage_risk",
    "known_at_origin",
]:
    feature_registry[col] = _to_bool(feature_registry[col])

KEY_COLUMNS = feature_registry.loc[feature_registry["role"] == "key", "column"].tolist()
TARGET_COLUMNS = feature_registry.loc[
    feature_registry["role"].isin(["target", "robustness_target"]),
    "column",
].tolist()

CATEGORICAL_FEATURE_CANDIDATES = [
    "AvdelingID",
    "AvdelingTekst",
    "Region",
    "Analyse_Kategori",
    "Ukedag",
    "HelligdagNavn",
    "season",
]

FORBIDDEN_FEATURE_COLUMNS = set(KEY_COLUMNS + TARGET_COLUMNS + ["origin_season"]) - {
    "AvdelingID",
    "Analyse_Kategori",
}


def feature_set_columns(label):
    flag = {
        "M1": "allowed_m1_baseline",
        "M2": "allowed_m2_synthetic_point_weather",
        "M3": "allowed_m3_perfect_information",
        "M4": "allowed_m4_synthetic_engineered_weather",
    }[label]
    reg = feature_registry.loc[feature_registry[flag], "column"].tolist()
    reg = [c for c in reg if c not in FORBIDDEN_FEATURE_COLUMNS]
    explicit = [c for c in CATEGORICAL_FEATURE_CANDIDATES if c in {"AvdelingID", "Analyse_Kategori"}]
    return list(dict.fromkeys(explicit + reg))


feature_sets = {label: feature_set_columns(label) for label in FEATURE_SETS}


def split_numeric_categorical(columns):
    cat = [c for c in columns if c in CATEGORICAL_FEATURE_CANDIDATES]
    num = [c for c in columns if c not in CATEGORICAL_FEATURE_CANDIDATES]
    return num, cat


feature_set_numeric = {}
feature_set_categorical = {}
for label, cols in feature_sets.items():
    n, c = split_numeric_categorical(cols)
    feature_set_numeric[label] = n
    feature_set_categorical[label] = c
    print(f"{label}: {len(cols)} features ({len(n)} numeric, {len(c)} categorical)")


## Pre-fit validation

This cell enforces the registry-level leakage rules before model fitting. M1, M2 and M4 must exclude all leakage-risk columns; M3 may include only realised-weather perfect-information columns. Target columns, forecast keys and disallowed weather-truth name patterns are rejected from operational feature sets.


In [ ]:
leakage_columns = set(feature_registry.loc[feature_registry["leakage_risk"], "column"])
realised_weather_columns = set(
    feature_registry.loc[
        feature_registry["feature_group"] == "realised_weather_perfect_information",
        "column",
    ]
)


def assert_no_leakage(label):
    cols = set(feature_sets[label])
    if label in {"M1", "M2", "M4"}:
        bad = cols & leakage_columns
        if bad:
            raise AssertionError(f"{label} contains leakage-risk columns: {sorted(bad)}")
    elif label == "M3":
        bad = (cols & leakage_columns) - realised_weather_columns
        if bad:
            raise AssertionError(
                f"M3 contains non-realised-weather leakage-risk columns: {sorted(bad)}"
            )
    overlap = cols & set(TARGET_COLUMNS)
    if overlap:
        raise AssertionError(f"{label} overlaps target columns: {sorted(overlap)}")
    key_overlap = cols & {"target_date", "origin_date", "horizon"}
    if key_overlap:
        raise AssertionError(f"{label} overlaps non-categorical key columns: {sorted(key_overlap)}")
    if "origin_season" in cols:
        raise AssertionError(
            f"{label} contains origin_season; must be diagnostic only."
        )


def assert_no_disallowed_substrings_m1_m2_m4():
    disallowed = (
        "_realised", "_observed", "_obs_", "_error", "_truth",
        "msl", "pressure", "apparent",
        "era5_", "_era5", "clim_", "_climatology",
    )
    for label in ["M1", "M2", "M4"]:
        bad = [c for c in feature_sets[label] if any(s in c.lower() for s in disallowed)]
        if bad:
            raise AssertionError(f"{label} contains disallowed columns by name pattern: {bad}")
        bad_truth = [c for c in feature_sets[label] if c in realised_weather_columns]
        if bad_truth:
            raise AssertionError(
                f"{label} contains realised-weather perfect-information columns: {bad_truth}"
            )


for label in FEATURE_SETS:
    assert_no_leakage(label)
assert_no_disallowed_substrings_m1_m2_m4()
print("Pre-fit leakage and substring assertions PASSED.")
print("Note: M3 is included with realised target-day weather as a non-operational benchmark.")


## Load tuned LightGBM parameters

This cell loads the validation-selected LightGBM parameters from notebook 08, removes helper-only tuning keys, and records the exact parameter payload used in the rolling-origin run.


In [ ]:
with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
    tuning_payload = json.load(f)

assert set(tuning_payload.get("feature_sets", {}).keys()) >= set(FEATURE_SETS), (
    f"Tuned best_params JSON is missing feature sets: required {FEATURE_SETS}, "
    f"got {sorted(tuning_payload.get('feature_sets', {}).keys())}"
)


def resolve_tuned_params(label):
    raw = dict(tuning_payload["feature_sets"][label]["best_params"])
    raw.pop("use_max_depth", None)
    if "max_depth" not in raw:
        raw["max_depth"] = -1
    return raw


TUNED_PARAMS = {label: resolve_tuned_params(label) for label in FEATURE_SETS}

# Reuse static LightGBM kwargs recorded by notebook 08 when available.
STATIC_LGBM_KWARGS = tuning_payload.get("lightgbm_static_kwargs", {})
STATIC_LGBM_KWARGS.setdefault("objective", "regression")
STATIC_LGBM_KWARGS.setdefault("n_jobs", N_JOBS)
STATIC_LGBM_KWARGS.setdefault("random_state", RANDOM_STATE)
STATIC_LGBM_KWARGS.setdefault("verbose", -1)


# Record the exact parameter payload used in this run.
used_params_payload = {
    "run_mode": RUN_MODE,
    "model_family": MODEL_FAMILY,
    "eval_start_date": EVAL_START_DATE,
    "eval_end_date": EVAL_END_DATE,
    "origin_step_days": ORIGIN_STEP_DAYS,
    "horizons": HORIZONS_TO_RUN,
    "feature_sets": FEATURE_SETS,
    "lightgbm_static_kwargs": STATIC_LGBM_KWARGS,
    "tuned_params": {label: TUNED_PARAMS[label] for label in FEATURE_SETS},
    "tuning_source": str(BEST_PARAMS_PATH.relative_to(PROJECT_ROOT).as_posix()),
}
USED_PARAMS_JSON_PATH.write_text(
    json.dumps(used_params_payload, indent=2, default=str),
    encoding="utf-8",
)
print(f"Saved used-params JSON: {project_relative(USED_PARAMS_JSON_PATH)}")

used_params_rows = []
for label in FEATURE_SETS:
    row = {
        "run_mode":    RUN_MODE,
        "feature_set": label,
    }
    row.update({f"static_{k}": v for k, v in STATIC_LGBM_KWARGS.items()})
    row.update({f"tuned_{k}": v for k, v in TUNED_PARAMS[label].items()})
    used_params_rows.append(row)
pd.DataFrame(used_params_rows).to_csv(USED_PARAMS_CSV_PATH, index=False)
print(f"Saved used-params CSV: {project_relative(USED_PARAMS_CSV_PATH)}")
print()
for label in FEATURE_SETS:
    print(f"{label} tuned params: {TUNED_PARAMS[label]}")


## Load ML panel and build the origin schedule

The panel is loaded with only the keys, target, selected feature columns and optional audit columns required for the configured run. The origin schedule is built from the reserved evaluation period, and a prefit audit records weekday, opening-status, category and runtime diagnostics before model fitting.

Rows with missing targets are dropped, negative targets are rejected, and scoring is restricted to the reserved evaluation window. Training rows are later restricted to `target_date < origin_date` for each origin.


In [ ]:
available_columns = set(pq.ParquetFile(ML_PANEL_PATH).schema_arrow.names)
OPTIONAL_AUDIT_COLUMNS = [
    "Closed",
    "is_open",
    "Ukedag",
    "Helligdager",
    "HelligdagNavn",
    "target_year",
    "target_month",
    "target_iso_week",
    "origin_datetime_utc",
    "origin_hour",
]
selected_columns = sorted(
    set(
        KEY_COLUMNS
        + ["Antall_total"]
        + [c for label in FEATURE_SETS for c in feature_sets[label]]
        + [c for c in OPTIONAL_AUDIT_COLUMNS if c in available_columns]
    )
)
missing_required = sorted((set(KEY_COLUMNS + ["Antall_total"]) - available_columns))
if missing_required:
    raise AssertionError(f"ML panel is missing required columns: {missing_required}")

panel = pd.read_parquet(ML_PANEL_PATH, columns=selected_columns)
panel["target_date"] = pd.to_datetime(panel["target_date"]).dt.normalize()
panel["origin_date"] = pd.to_datetime(panel["origin_date"]).dt.normalize()
panel["horizon"] = panel["horizon"].astype("int8")
panel = panel.loc[panel["horizon"].isin(HORIZONS_TO_RUN)].reset_index(drop=True)
panel = panel.loc[panel["Antall_total"].notna()].reset_index(drop=True)
if (panel["Antall_total"] < 0).any():
    raise AssertionError("Antall_total contains negative values; check 06 panel build.")
if "origin_season" in panel.columns:
    panel = panel.drop(columns=["origin_season"])

eval_start_ts = pd.Timestamp(EVAL_START_DATE).normalize()
eval_end_ts = pd.Timestamp(EVAL_END_DATE).normalize()


def build_origin_schedule(panel_, eval_start, eval_end, mode, step_days, max_origins=None):
    available = sorted(
        pd.to_datetime(
            panel_.loc[
                (panel_["origin_date"] >= eval_start) & (panel_["origin_date"] <= eval_end),
                "origin_date",
            ]
            .dropna()
            .unique()
        ).normalize()
    )
    available_set = set(available)
    if mode == "daily":
        schedule = [d for d in pd.date_range(eval_start, eval_end, freq="D") if d in available_set]
    elif mode == "step":
        schedule = []
        cursor = eval_start
        while cursor <= eval_end:
            if cursor in available_set:
                schedule.append(cursor)
            cursor = cursor + pd.Timedelta(days=int(step_days))
    elif mode == "balanced_weekday":
        if max_origins is None:
            schedule = available
        else:
            counts = {i: 0 for i in range(7)}
            schedule = []
            for d in available:
                wd = int(d.weekday())
                if counts[wd] <= min(counts.values()) or len(schedule) < 7:
                    schedule.append(d)
                    counts[wd] += 1
                if len(schedule) >= max_origins:
                    break
    else:
        raise ValueError(f"Unsupported ORIGIN_SCHEDULE_MODE: {mode!r}")
    return schedule[:int(max_origins)] if max_origins is not None else schedule


origin_schedule = build_origin_schedule(panel, eval_start_ts, eval_end_ts, ORIGIN_SCHEDULE_MODE, ORIGIN_STEP_DAYS, MAX_ORIGINS)


def _weekday_label(series):
    return pd.to_datetime(series).dt.day_name()


def _open_status_frame(df):
    if "is_open" in df.columns:
        return pd.Series(np.where(df["is_open"].astype("float64") > 0, "open", "closed"), index=df.index, name="open_status")
    if "Closed" in df.columns:
        return pd.Series(np.where(df["Closed"].astype("float64") > 0, "closed", "open"), index=df.index, name="open_status")
    return None


def run_preflight_audit(panel_, origin_schedule_, horizons_, feature_sets_, run_mode):
    PREFLIGHT_AUDIT_DIR.mkdir(parents=True, exist_ok=True)
    sched = pd.DataFrame({"origin_date": origin_schedule_})
    if not sched.empty:
        sched["origin_weekday"] = _weekday_label(sched["origin_date"])
    sched["run_mode"] = run_mode
    sched["origin_schedule_mode"] = ORIGIN_SCHEDULE_MODE
    sched["origin_step_days"] = ORIGIN_STEP_DAYS if ORIGIN_SCHEDULE_MODE == "step" else np.nan
    sched["n_origins"] = len(origin_schedule_)
    sched["n_horizons"] = len(horizons_)
    sched["n_feature_sets"] = len(feature_sets_)
    sched.to_csv(PREFLIGHT_AUDIT_DIR / f"origin_schedule_summary_{run_mode}.csv", index=False)

    block = panel_.loc[
        panel_["origin_date"].isin(origin_schedule_)
        & panel_["horizon"].isin(horizons_)
        & (panel_["target_date"] <= eval_end_ts)
    ].copy()
    warnings_list = []
    if block.empty:
        warnings_list.append("No evaluation rows found for the configured origin schedule.")
        target_weekday = pd.DataFrame(columns=["horizon", "target_weekday", "n_rows", "share"])
        origin_weekday = pd.DataFrame(columns=["origin_weekday", "n_origins", "share"])
        target_summary = pd.DataFrame(columns=["horizon", "n_rows", "mean_antall_total"])
        zero_sales = pd.DataFrame(columns=["horizon", "n_rows", "zero_sales_share"])
        category_dist = pd.DataFrame(columns=["horizon", "Analyse_Kategori", "n_rows", "share"])
        open_closed = pd.DataFrame()
    else:
        block["target_weekday"] = _weekday_label(block["target_date"])
        block["origin_weekday"] = _weekday_label(block["origin_date"])
        block["zero_sales"] = block["Antall_total"].astype("float64") <= 0
        target_weekday = block.groupby(["horizon", "target_weekday"], observed=True).size().reset_index(name="n_rows")
        target_weekday["share"] = target_weekday["n_rows"] / target_weekday.groupby(
            "horizon", observed=True
        )["n_rows"].transform("sum")
        origin_weekday = (
            sched.groupby("origin_weekday", observed=True).size().reset_index(name="n_origins")
            if "origin_weekday" in sched.columns
            else pd.DataFrame()
        )
        if not origin_weekday.empty:
            origin_weekday["share"] = origin_weekday["n_origins"] / origin_weekday["n_origins"].sum()
        target_summary = (
            block.groupby("horizon", observed=True)["Antall_total"]
            .agg(
                n_rows="count",
                mean_antall_total="mean",
                median_antall_total="median",
                std_antall_total="std",
                min_antall_total="min",
                max_antall_total="max",
            )
            .reset_index()
        )
        zero_sales = (
            block.groupby("horizon", observed=True)
            .agg(n_rows=("Antall_total", "size"), zero_sales_share=("zero_sales", "mean"))
            .reset_index()
        )
        category_dist = block.groupby(["horizon", "Analyse_Kategori"], observed=True).size().reset_index(name="n_rows")
        category_dist["share"] = category_dist["n_rows"] / category_dist.groupby(
            "horizon", observed=True
        )["n_rows"].transform("sum")
        status = _open_status_frame(block)
        if status is not None:
            block["open_status"] = status
            open_closed = block.groupby(["horizon", "open_status"], observed=True).size().reset_index(name="n_rows")
            open_closed["share"] = open_closed["n_rows"] / open_closed.groupby(
                "horizon", observed=True
            )["n_rows"].transform("sum")
        else:
            open_closed = pd.DataFrame()
        min_wd = 5 if run_mode == "full" else 3
        wd_n = target_weekday.loc[target_weekday["n_rows"] > 0].groupby("horizon", observed=True)["target_weekday"].nunique()
        for h in horizons_:
            n_unique = int(wd_n.get(h, 0))
            if n_unique < min_wd:
                warnings_list.append(
                    f"h={h} has only {n_unique} unique target weekdays; threshold={min_wd}."
                )
        for h in [3, 10]:
            sub = target_weekday.loc[target_weekday["horizon"] == h]
            if not sub.empty:
                sunday = float(sub.loc[sub["target_weekday"] == "Sunday", "share"].sum())
                if sunday > 0.60:
                    warnings_list.append(f"h={h} target rows are Sunday-dominated: share={sunday:.3f}.")
                if float(sub["share"].max()) >= 0.95:
                    warnings_list.append(
                        f"h={h} is effectively a single-weekday horizon: "
                        f"max_share={float(sub['share'].max()):.3f}."
                    )
        if not open_closed.empty:
            for h in [3, 10]:
                sub = open_closed.loc[open_closed["horizon"] == h]
                closed = float(sub.loc[sub["open_status"] == "closed", "share"].sum()) if not sub.empty else 0.0
                if closed > 0.60:
                    warnings_list.append(f"h={h} target rows are closed-day dominated: share={closed:.3f}.")
        if not zero_sales.empty:
            zr = float(zero_sales["zero_sales_share"].max() - zero_sales["zero_sales_share"].min())
            if zr > 0.15:
                warnings_list.append(f"Zero-sales share differs strongly across horizons: range={zr:.3f}.")

    target_weekday.to_csv(PREFLIGHT_AUDIT_DIR / f"target_weekday_distribution_by_horizon_{run_mode}.csv", index=False)
    origin_weekday.to_csv(PREFLIGHT_AUDIT_DIR / f"origin_weekday_distribution_{run_mode}.csv", index=False)
    target_summary.to_csv(PREFLIGHT_AUDIT_DIR / f"target_summary_by_horizon_{run_mode}.csv", index=False)
    zero_sales.to_csv(PREFLIGHT_AUDIT_DIR / f"zero_sales_share_by_horizon_{run_mode}.csv", index=False)
    category_dist.to_csv(PREFLIGHT_AUDIT_DIR / f"category_distribution_by_horizon_{run_mode}.csv", index=False)
    if not open_closed.empty:
        open_closed.to_csv(PREFLIGHT_AUDIT_DIR / f"open_closed_distribution_by_horizon_{run_mode}.csv", index=False)

    nfits = int(len(origin_schedule_) * len(horizons_) * len(feature_sets_))
    hours = nfits * EST_MINUTES_PER_FIT / 60.0
    if hours > 36:
        warnings_list.append(f"Estimated runtime exceeds 36 hours: {hours:.2f} hours.")
    if run_mode == "full" and ORIGIN_SCHEDULE_MODE == "step" and ORIGIN_STEP_DAYS != 5 and not ALLOW_NON_STEP5_FULL:
        warnings_list.append("Full mode is configured with a non-step5 origin schedule.")
    hard = []
    if run_mode == "full" and REQUIRE_PREFLIGHT_PASS_FOR_FULL:
        hard = [
            w
            for w in warnings_list
            if (
                "unique target weekdays" in w
                or "single-weekday" in w
                or "non-step5" in w
                or "No evaluation rows" in w
            )
        ]
    runtime = pd.DataFrame(
        [
            {
                "run_mode": run_mode,
                "origin_schedule_mode": ORIGIN_SCHEDULE_MODE,
                "origin_step_days": ORIGIN_STEP_DAYS if ORIGIN_SCHEDULE_MODE == "step" else np.nan,
                "n_origins": len(origin_schedule_),
                "n_horizons": len(horizons_),
                "n_feature_sets": len(feature_sets_),
                "estimated_n_fits": nfits,
                "est_minutes_per_fit": EST_MINUTES_PER_FIT,
                "estimated_runtime_hours": hours,
                "preflight_warning_count": len(warnings_list),
                "preflight_hard_fail_count": len(hard),
                "warnings": " | ".join(warnings_list),
                "hard_fail_reasons": " | ".join(hard),
            }
        ]
    )
    runtime.to_csv(PREFLIGHT_AUDIT_DIR / f"estimated_runtime_{run_mode}.csv", index=False)
    print(f"Pre-flight audit saved under {project_relative(PREFLIGHT_AUDIT_DIR)}")
    print(f"Estimated fits: {nfits:,}")
    print(f"Estimated runtime: {hours:.2f} hours at {EST_MINUTES_PER_FIT:.2f} minutes/fit")
    if warnings_list:
        print("Pre-flight warnings:")
        for w in warnings_list:
            print(f"  - {w}")
    else:
        print("Pre-flight warnings: none")
    return {"warnings": warnings_list, "hard_fail_reasons": hard, "passed_for_full": len(hard) == 0}


preflight_result = run_preflight_audit(panel, origin_schedule, HORIZONS_TO_RUN, FEATURE_SETS, RUN_MODE)
if RUN_MODE == "full" and REQUIRE_PREFLIGHT_PASS_FOR_FULL and not preflight_result["passed_for_full"]:
    raise AssertionError(
        "Full rolling-origin evaluation stopped by pre-flight go/no-go check: "
        + " | ".join(preflight_result["hard_fail_reasons"])
    )
if RUN_MODE == "full" and ORIGIN_SCHEDULE_MODE == "step" and ORIGIN_STEP_DAYS != 5 and not ALLOW_NON_STEP5_FULL:
    raise AssertionError("Full mode must use ORIGIN_STEP_DAYS = 5 unless ALLOW_NON_STEP5_FULL=True.")

print(f"Panel rows after horizon and target filter: {len(panel):,}")
print(f"Origin schedule length: {len(origin_schedule)}")
print(f"First origin: {origin_schedule[0].date() if origin_schedule else 'n/a'}")
print(f"Last origin:  {origin_schedule[-1].date() if origin_schedule else 'n/a'}")
panel = panel.sort_values(["target_date", "horizon"]).reset_index(drop=True)


## Pipeline builder and metric functions

This cell defines the preprocessing pipeline and forecast-error metrics used for all feature sets. Numeric variables pass through unchanged, categorical variables are one-hot encoded with unknown levels ignored, and metrics are computed on the original sales scale.


In [ ]:
def build_pipeline(numeric_cols, categorical_cols, tuned_params):
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    transformer = ColumnTransformer(
        transformers=[
            ("num", "passthrough", numeric_cols),
            ("cat", encoder, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    full_params = {**STATIC_LGBM_KWARGS, **tuned_params}
    estimator = LGBMRegressor(**full_params)
    return Pipeline([("preprocess", transformer), ("model", estimator)])


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.mean(np.abs(y_true - y_pred)))


def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    denom = float(np.sum(np.abs(y_true)))
    if denom <= 0:
        return float("nan")
    return float(np.sum(np.abs(y_true - y_pred)) / denom)


def bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    return float(np.mean(y_pred - y_true))


def all_metrics(y_true, y_pred):
    return {
        "rmse": rmse(y_true, y_pred),
        "mae":  mae(y_true, y_pred),
        "wape": wape(y_true, y_pred),
        "bias": bias(y_true, y_pred),
        "n":    int(len(y_true)),
    }


def horizon_tier_for(h):
    h_int = int(h)
    if h_int in (0, 1, 2):
        return "actual_meps_h0_h2"
    if h_int in (3, 4, 5, 6, 7, 8, 9, 10):
        return "synthetic_scenario_h3_h10"
    return f"unsupported_h{h_int}"


## Rolling-origin loop

The loop fits one LightGBM model for each origin, feature set and horizon. Training rows satisfy `target_date < origin_date`; test rows belong to the current origin and do not exceed the evaluation end date. Predictions are clipped to non-negative values when configured, and fit-level timing, row counts and status information are logged.


In [ ]:
KEY_PRED_COLUMNS = ["AvdelingID", "Analyse_Kategori", "origin_date", "target_date", "horizon"]

prediction_frames = []
runtime_rows = []
missing_combinations = []

t_start = time.time()

for origin_date in origin_schedule:
    print(f"\n=== Origin {origin_date.date()} ===")
    # Candidate training rows; each horizon is filtered inside the inner loop.
    train_panel_origin = panel.loc[panel["target_date"] < origin_date]
    if train_panel_origin.empty:
        print("  No training rows available; skipping origin.")
        for label in FEATURE_SETS:
            for h in HORIZONS_TO_RUN:
                missing_combinations.append(
                    {
                        "origin_date": origin_date.date().isoformat(),
                        "feature_set": label,
                        "horizon": int(h),
                        "reason": "no_training_rows",
                    }
                )
        continue

    # Rows scored for the current forecast origin.
    test_panel_origin = panel.loc[
        (panel["origin_date"] == origin_date)
        & (panel["target_date"] <= eval_end_ts)
    ]
    if test_panel_origin.empty:
        print("  No test rows for this origin; skipping.")
        for label in FEATURE_SETS:
            for h in HORIZONS_TO_RUN:
                missing_combinations.append(
                    {
                        "origin_date": origin_date.date().isoformat(),
                        "feature_set": label,
                        "horizon": int(h),
                        "reason": "no_test_rows",
                    }
                )
        continue

    for label in FEATURE_SETS:
        numeric_cols = feature_set_numeric[label]
        categorical_cols = feature_set_categorical[label]
        feature_cols = numeric_cols + categorical_cols
        tuned            = TUNED_PARAMS[label]

        for h in HORIZONS_TO_RUN:
            train_h = train_panel_origin.loc[train_panel_origin["horizon"] == h]
            test_h  = test_panel_origin.loc[test_panel_origin["horizon"] == h]

            if len(train_h) == 0 or len(test_h) == 0:
                missing_combinations.append({
                    "origin_date": origin_date.date().isoformat(),
                    "feature_set": label,
                    "horizon": int(h),
                    "reason": ("no_train" if len(train_h) == 0 else "no_test"),
                })
                continue

            X_train = train_h[feature_cols]
            y_train = train_h["Antall_total"].astype("float32").to_numpy()
            X_test  = test_h[feature_cols]
            y_test  = test_h["Antall_total"].astype("float32").to_numpy()

            pipeline = build_pipeline(numeric_cols, categorical_cols, tuned)
            t0 = time.time()
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    pipeline.fit(X_train, y_train)
                training_time = time.time() - t0
                t1 = time.time()
                y_pred = np.asarray(pipeline.predict(X_test), dtype="float64")
                prediction_time = time.time() - t1
            except Exception as exc:
                runtime_rows.append({
                    "run_mode": RUN_MODE,
                    "origin_date": origin_date.date().isoformat(),
                    "feature_set": label,
                    "horizon": int(h),
                    "train_rows": int(len(train_h)),
                    "test_rows":  int(len(test_h)),
                    "training_time_seconds":   float("nan"),
                    "prediction_time_seconds": float("nan"),
                    "n_clipped_predictions":   0,
                    "share_clipped_predictions": float("nan"),
                    "status": "failed",
                    "error":  f"{type(exc).__name__}: {exc}",
                })
                print(f"  [{label} h={h}] FAILED: {type(exc).__name__}: {exc}")
                continue

            if CLIP_NEGATIVE_PREDICTIONS:
                n_clipped = int((y_pred < 0).sum())
                y_pred = np.clip(y_pred, 0.0, None)
            else:
                n_clipped = 0
            share_clipped = n_clipped / max(len(y_pred), 1)

            if not np.all(np.isfinite(y_pred)):
                raise AssertionError(
                    f"Non-finite predictions for origin={origin_date.date()} "
                    f"feature_set={label} horizon={h}"
                )

            pred_frame = test_h[KEY_PRED_COLUMNS].copy()
            pred_frame["run_mode"]      = RUN_MODE
            pred_frame["model_family"]  = MODEL_FAMILY
            pred_frame["feature_set"]   = label
            pred_frame["y_true"]        = y_test
            pred_frame["y_pred"]        = y_pred
            prediction_frames.append(pred_frame)

            runtime_rows.append({
                "run_mode": RUN_MODE,
                "origin_date": origin_date.date().isoformat(),
                "feature_set": label,
                "horizon": int(h),
                "train_rows": int(len(train_h)),
                "test_rows":  int(len(test_h)),
                "training_time_seconds":     training_time,
                "prediction_time_seconds":   prediction_time,
                "n_clipped_predictions":     n_clipped,
                "share_clipped_predictions": share_clipped,
                "status": "ok",
                "error":  "",
            })

            print(
                f"  [{label} h={h}] train={len(train_h):,} test={len(test_h):,} "
                f"fit={training_time:.1f}s pred={prediction_time:.2f}s "
                f"clip_share={share_clipped:.4f}"
            )
            gc.collect()

total_wall_minutes = (time.time() - t_start) / 60.0
print(f"\nRolling-origin wall time: {total_wall_minutes:.2f} min")


## Prediction output

Prediction rows are concatenated and written with forecast keys, feature-set metadata, observed and predicted sales, row-level errors and available audit columns.


In [ ]:
if prediction_frames:
    predictions_df = pd.concat(prediction_frames, ignore_index=True)
else:
    predictions_df = pd.DataFrame(
        columns=[
            *KEY_PRED_COLUMNS,
            "run_mode",
            "model_family",
            "feature_set",
            "y_true",
            "y_pred",
        ]
    )

# Label archived-forecast and synthetic-emulator horizon tiers.
if not predictions_df.empty and "horizon" in predictions_df.columns:
    predictions_df["horizon_tier"] = predictions_df["horizon"].map(horizon_tier_for)

PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
predictions_df.to_parquet(PREDICTIONS_PATH, index=False)
print(f"Saved: {project_relative(PREDICTIONS_PATH)} ({len(predictions_df):,} rows)")


## Prediction validation checks

This cell verifies prediction-key uniqueness, leakage constraints, temporal boundaries, horizon coverage, finite predictions, clipping logs, parameter logging and output-suffix isolation.


In [ ]:
validation_check_rows = []


def _record_check(name, passed, detail=""):
    validation_check_rows.append(
        {
            "run_mode": RUN_MODE,
            "check": name,
            "passed": bool(passed),
            "detail": detail,
        }
    )


# Unique row per prediction key.
if not predictions_df.empty:
    dup_cols = ["AvdelingID", "Analyse_Kategori", "origin_date", "target_date", "horizon", "feature_set"]
    n_dups = int(predictions_df.duplicated(subset=dup_cols).sum())
    _record_check("no_duplicate_prediction_keys", n_dups == 0, f"duplicates={n_dups}")
else:
    _record_check("no_duplicate_prediction_keys", True, "predictions_df empty")

# Registry-level leakage constraints.
for label in ["M1", "M2", "M4"]:
    cols = set(feature_sets[label])
    bad = cols & leakage_columns
    _record_check(f"{label}_no_leakage_columns", len(bad) == 0, f"bad={sorted(bad)}")

# M3 leakage is limited to realised-weather benchmark columns.
m3_bad = (set(feature_sets["M3"]) & leakage_columns) - realised_weather_columns
_record_check("M3_only_realised_weather_leakage", len(m3_bad) == 0, f"bad={sorted(m3_bad)}")

# Training rows must precede each forecast origin.
training_leak_violations = 0
for origin_date in origin_schedule[:5]:  # small sample to keep this cheap
    sub = panel.loc[panel["target_date"] < origin_date]
    if (sub["target_date"] >= origin_date).any():
        training_leak_violations += 1
_record_check(
    "training_window_strict_lt_origin",
    training_leak_violations == 0,
    f"violations_in_sample={training_leak_violations}",
)

# Origin dates must fall inside the evaluation period.
if not predictions_df.empty:
    bad = (predictions_df["origin_date"] < eval_start_ts).sum()
    _record_check("test_origin_after_start", bad == 0, f"violations={int(bad)}")
else:
    _record_check("test_origin_after_start", True, "predictions_df empty")

# Target dates must not exceed the evaluation boundary.
if not predictions_df.empty:
    bad = (predictions_df["target_date"] > eval_end_ts).sum()
    _record_check("test_target_before_end", bad == 0, f"violations={int(bad)}")
else:
    _record_check("test_target_before_end", True, "predictions_df empty")

# Report missing horizon and feature-set combinations without failing.
expected_horizons = set(HORIZONS_TO_RUN)
present_horizons = (
    set(predictions_df["horizon"].unique().tolist()) if not predictions_df.empty else set()
)
_record_check(
    "all_horizons_present",
    present_horizons == expected_horizons,
    f"missing={sorted(expected_horizons - present_horizons)}",
)

# Predictions must be finite.
if not predictions_df.empty:
    non_finite = int((~np.isfinite(predictions_df["y_pred"].to_numpy())).sum())
    _record_check("predictions_finite", non_finite == 0, f"non_finite={non_finite}")
else:
    _record_check("predictions_finite", True, "predictions_df empty")

# Clipping logs must exist for fitted models.
if not predictions_df.empty:
    runtime_df_local = pd.DataFrame(runtime_rows)
    max_clip_share = (
        float(runtime_df_local.loc[runtime_df_local["status"] == "ok", "share_clipped_predictions"].max())
        if not runtime_df_local.empty else float("nan")
    )
    _record_check("clip_share_recorded", True, f"max_clip_share={max_clip_share:.6f}")
else:
    _record_check("clip_share_recorded", True, "predictions_df empty")

# Parameters from notebook 08 must be present in the run log.
_record_check(
    "tuned_params_used_and_logged",
    USED_PARAMS_JSON_PATH.exists() and USED_PARAMS_CSV_PATH.exists(),
    f"json={USED_PARAMS_JSON_PATH.exists()} csv={USED_PARAMS_CSV_PATH.exists()}",
)

# Run outputs must include FILE_SUFFIX.
suffix_ok = all(str(p).endswith(f"{FILE_SUFFIX}.csv") or
                str(p).endswith(f"{FILE_SUFFIX}.parquet") or
                str(p).endswith(f"{FILE_SUFFIX}.json") or
                str(p).endswith(f"{FILE_SUFFIX}.md") or
                str(p).endswith(f"{FILE_SUFFIX}.html")
                for p in [
                    PREDICTIONS_PATH, METRICS_PATH, GAIN_SUMMARY_PATH, WEATHER_VALUE_DECOMP_PATH,
                    M4_VS_M2_DIAG_PATH, HORIZON_TIER_PATH, RUNTIME_SUMMARY_PATH,
                    VALIDATION_CHECKS_PATH, FEATURE_SET_SUMMARY_PATH,
                    USED_PARAMS_JSON_PATH, USED_PARAMS_CSV_PATH,
                ])
_record_check("smoke_full_suffix_isolation", suffix_ok, f"FILE_SUFFIX={FILE_SUFFIX!r}")

validation_checks_df = pd.DataFrame(validation_check_rows)
validation_checks_df.to_csv(VALIDATION_CHECKS_PATH, index=False)
print(f"Saved: {project_relative(VALIDATION_CHECKS_PATH)}")
print(validation_checks_df.to_string(index=False))


## Metrics and comparison summaries

Metrics are computed on the original sales scale at pooled horizon, horizon-tier and category-horizon levels. Pairwise comparison tables report how each comparison feature set differs from its reference feature set; positive improvement values indicate lower error for the comparison model.


In [ ]:
metric_rows = []

if not predictions_df.empty:
    # Pooled feature-set by horizon metrics.
    for (fs, h), block in predictions_df.groupby(["feature_set", "horizon"], observed=True):
        m = all_metrics(block["y_true"].to_numpy(), block["y_pred"].to_numpy())
        metric_rows.append(
            {
                "run_mode": RUN_MODE,
                "scope": "feature_set_horizon",
                "feature_set": fs,
                "horizon": int(h),
                "horizon_tier": horizon_tier_for(h),
                "Analyse_Kategori": "__POOLED__",
                **m,
            }
        )

    # Pooled feature-set by horizon-tier metrics.
    for (fs, tier), block in predictions_df.groupby(["feature_set", "horizon_tier"], observed=True):
        m = all_metrics(block["y_true"].to_numpy(), block["y_pred"].to_numpy())
        metric_rows.append(
            {
                "run_mode": RUN_MODE,
                "scope": "feature_set_tier",
                "feature_set": fs,
                "horizon": -1,
                "horizon_tier": tier,
                "Analyse_Kategori": "__POOLED__",
                **m,
            }
        )

    # Category and horizon metrics by feature set.
    for (fs, cat, h), block in predictions_df.groupby(
        ["feature_set", "Analyse_Kategori", "horizon"], observed=True
    ):
        m = all_metrics(block["y_true"].to_numpy(), block["y_pred"].to_numpy())
        metric_rows.append(
            {
                "run_mode": RUN_MODE,
                "scope": "feature_set_category_horizon",
                "feature_set": fs,
                "horizon": int(h),
                "horizon_tier": horizon_tier_for(h),
                "Analyse_Kategori": cat,
                **m,
            }
        )

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(METRICS_PATH, index=False)
print(f"Saved: {project_relative(METRICS_PATH)} ({len(metrics_df):,} rows)")


In [ ]:
# Pairwise feature-set comparisons.
def _build_pair_table(metrics_df_, ref_fs, comp_fs):
    pivot_cols = ["scope", "horizon", "horizon_tier", "Analyse_Kategori"]
    metric_cols = ["rmse", "mae", "wape"]
    ref = metrics_df_.loc[metrics_df_["feature_set"] == ref_fs, pivot_cols + metric_cols].rename(
        columns={c: f"ref_{c}" for c in metric_cols}
    )
    comp = metrics_df_.loc[metrics_df_["feature_set"] == comp_fs, pivot_cols + metric_cols].rename(
        columns={c: f"comp_{c}" for c in metric_cols}
    )
    merged = ref.merge(comp, on=pivot_cols, how="inner")
    rows = []
    for metric in metric_cols:
        r = merged[f"ref_{metric}"].astype("float64")
        c = merged[f"comp_{metric}"].astype("float64")
        improvement = r - c
        with np.errstate(divide="ignore", invalid="ignore"):
            pct = np.where(r.abs() > 0, improvement / r, np.nan)
        block = merged[pivot_cols].copy()
        block["run_mode"] = RUN_MODE
        block["reference_feature_set"] = ref_fs
        block["comparison_feature_set"] = comp_fs
        block["metric"] = metric
        block["reference_value"] = r.values
        block["comparison_value"] = c.values
        block["metric_delta"] = (c - r).values
        block["metric_improvement"] = improvement.values
        block["pct_improvement"] = pct
        rows.append(block)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


pairs = [
    ("M1", "M2"),
    ("M1", "M3"),
    ("M1", "M4"),
    ("M2", "M4"),
    ("M2", "M3"),
    ("M4", "M3"),
]
gain_summary_rows = []
decomposition_blocks = []
for ref_fs, comp_fs in pairs:
    block = _build_pair_table(metrics_df, ref_fs, comp_fs)
    if not block.empty:
        decomposition_blocks.append(block)

weather_value_decomposition_df = (
    pd.concat(decomposition_blocks, ignore_index=True) if decomposition_blocks else pd.DataFrame()
)
weather_value_decomposition_df.to_csv(WEATHER_VALUE_DECOMPOSITION_PATH := WEATHER_VALUE_DECOMP_PATH, index=False)
print(f"Saved: {project_relative(WEATHER_VALUE_DECOMP_PATH)} "
      f"({len(weather_value_decomposition_df):,} rows)")


# M1-reference gain summary on pooled horizon metrics.
gain_rows = []
if not metrics_df.empty:
    pivot_cols = ["scope", "horizon", "horizon_tier", "Analyse_Kategori"]
    base = metrics_df.loc[
        (metrics_df["scope"] == "feature_set_horizon") & (metrics_df["feature_set"] == "M1"),
        pivot_cols + ["rmse", "mae", "wape"],
    ].rename(columns={"rmse": "rmse_M1", "mae": "mae_M1", "wape": "wape_M1"})
    other = metrics_df.loc[
        (metrics_df["scope"] == "feature_set_horizon") & (metrics_df["feature_set"].isin(["M2", "M3", "M4"])),
        pivot_cols + ["feature_set", "rmse", "mae", "wape"],
    ]
    merged = other.merge(base, on=pivot_cols, how="left")
    merged["delta_rmse"] = merged["rmse"] - merged["rmse_M1"]
    merged["delta_mae"]  = merged["mae"]  - merged["mae_M1"]
    merged["delta_wape"] = merged["wape"] - merged["wape_M1"]
    merged["pct_improvement_rmse"] = -merged["delta_rmse"] / merged["rmse_M1"]
    merged["pct_improvement_mae"]  = -merged["delta_mae"]  / merged["mae_M1"]
    merged["pct_improvement_wape"] = -merged["delta_wape"] / merged["wape_M1"]
    merged.insert(0, "run_mode", RUN_MODE)
    gain_summary_df = merged
else:
    gain_summary_df = pd.DataFrame()
gain_summary_df.to_csv(GAIN_SUMMARY_PATH, index=False)
print(f"Saved: {project_relative(GAIN_SUMMARY_PATH)} ({len(gain_summary_df):,} rows)")


In [ ]:
# M4-vs-M2 diagnostic table.
m4_vs_m2_rows = []
if not metrics_df.empty:
    pivot_cols = ["scope", "horizon", "horizon_tier", "Analyse_Kategori"]
    metric_cols = ["rmse", "mae", "wape"]
    m2 = metrics_df.loc[
        (metrics_df["scope"] == "feature_set_horizon") & (metrics_df["feature_set"] == "M2"),
        pivot_cols + metric_cols,
    ].rename(columns={c: f"m2_{c}" for c in metric_cols})
    m4 = metrics_df.loc[
        (metrics_df["scope"] == "feature_set_horizon") & (metrics_df["feature_set"] == "M4"),
        pivot_cols + metric_cols,
    ].rename(columns={c: f"m4_{c}" for c in metric_cols})
    merged = m2.merge(m4, on=pivot_cols, how="inner")
    for c in metric_cols:
        merged[f"m4_improvement_{c}"] = merged[f"m2_{c}"] - merged[f"m4_{c}"]


    def _support(row):
        d = [row["m4_improvement_rmse"], row["m4_improvement_mae"], row["m4_improvement_wape"]]
        positives = sum(1 for v in d if v > 1e-9)
        negatives = sum(1 for v in d if v < -1e-9)
        if positives == 3:
            return "strong_support"
        if negatives == 3:
            return "no_support"
        if positives >= 2:
            return "moderate_support"
        if positives >= 1 and negatives >= 1:
            return "mixed"
        return "weak_support"


    merged["support_label"] = merged.apply(_support, axis=1)
    merged.insert(0, "run_mode", RUN_MODE)
    m4_vs_m2_diagnostic_df = merged
else:
    m4_vs_m2_diagnostic_df = pd.DataFrame()
m4_vs_m2_diagnostic_df.to_csv(M4_VS_M2_DIAG_PATH, index=False)
print(f"Saved: {project_relative(M4_VS_M2_DIAG_PATH)} ({len(m4_vs_m2_diagnostic_df):,} rows)")


# Pooled horizon-tier metrics.
horizon_tier_summary_df = metrics_df.loc[metrics_df["scope"] == "feature_set_tier"].copy() \
    if not metrics_df.empty else pd.DataFrame()
horizon_tier_summary_df.to_csv(HORIZON_TIER_PATH, index=False)
print(f"Saved: {project_relative(HORIZON_TIER_PATH)} ({len(horizon_tier_summary_df):,} rows)")


# Fit-level runtime records.
runtime_df = pd.DataFrame(runtime_rows)
runtime_df.to_csv(RUNTIME_SUMMARY_PATH, index=False)
print(f"Saved: {project_relative(RUNTIME_SUMMARY_PATH)} ({len(runtime_df):,} rows)")


# Feature-set composition summary.
fss_rows = []
for label in FEATURE_SETS:
    cols = feature_sets[label]
    fss_rows.append({
        "run_mode":      RUN_MODE,
        "feature_set":   label,
        "is_operational": label != "M3",
        "n_features":    len(cols),
        "n_numeric":     len(feature_set_numeric[label]),
        "n_categorical": len(feature_set_categorical[label]),
        "feature_columns": ";".join(cols),
        "note": ("non-operational realised-weather point-information benchmark"
                 if label == "M3"
                 else "operational feature set"),
    })
feature_set_summary_df = pd.DataFrame(fss_rows)
feature_set_summary_df.to_csv(FEATURE_SET_SUMMARY_PATH, index=False)
print(f"Saved: {project_relative(FEATURE_SET_SUMMARY_PATH)}")

# Missing origin, feature-set and horizon combinations.
if missing_combinations:
    missing_df = pd.DataFrame(missing_combinations)
    out_missing = OUTPUT_ROLLING_DIR / f"lightgbm_rolling_origin_missing_combinations{FILE_SUFFIX}.csv"
    missing_df.to_csv(out_missing, index=False)
    print(f"Saved (informative): {project_relative(out_missing)} "
          f"({len(missing_df)} rows)")


## Optional HTML and Markdown report

The optional report summarizes pooled metrics and the main feature-set comparisons in a thesis-readable format. It labels M3 as non-operational and distinguishes archived MEPS horizons h=0-h=2 from synthetic-emulator horizons h=3-h=10.


In [ ]:
def _fmt_rmse(v):
    return f"{v:.3f}" if pd.notna(v) and np.isfinite(v) else ""


def _fmt_pct(v):
    return f"{v*100:.2f}%" if pd.notna(v) and np.isfinite(v) else ""


def _table_css():
    return """
<style>
  body { font-family: \"Times New Roman\", Times, serif; font-size: 9.5pt; margin: 18px; }
  h1 { font-size: 14pt; text-align: center; margin: 0 0 8px 0; }
  h2 { font-size: 11pt; margin: 16px 0 4px 0; }
  table { border-collapse: collapse; border-top: 1.2px solid #000; border-bottom: 1.2px solid #000; width: 100%; }
  th, td { padding: 3px 6px; vertical-align: top; }
  thead th { text-align: center; font-weight: bold; }
  thead tr:last-child th { border-bottom: 1px solid #000; }
  tbody td { text-align: center; white-space: nowrap; }
  tbody td:first-child { text-align: left; }
  .note { margin-top: 6px; font-size: 9.5pt; }
</style>
"""


def _render_metric_pivot(metrics_df_):
    if metrics_df_.empty:
        return pd.DataFrame()
    pooled_fh = metrics_df_.loc[metrics_df_["scope"] == "feature_set_horizon"].copy()
    pooled_fh["rmse_fmt"] = pooled_fh["rmse"].map(_fmt_rmse)
    pivot = pooled_fh.pivot_table(
        index=["feature_set"],
        columns="horizon",
        values="rmse",
        aggfunc="mean",
    ).round(3)
    return pivot


pivot_rmse_df = _render_metric_pivot(metrics_df)

# Markdown report body.
def _to_pipe_table(df, float_fmt="{:.3f}"):
    if df is None or df.empty:
        return "(empty)"
    headers = [str(c) for c in df.columns]
    out = []
    out.append("| " + " | ".join([df.index.name or "index", *headers]) + " |")
    out.append("| " + " | ".join(["---"] * (len(headers) + 1)) + " |")
    for idx, row in df.iterrows():
        cells = [str(idx)] + [
            float_fmt.format(v) if isinstance(v, (int, float)) and pd.notna(v) and np.isfinite(v) else ""
            for v in row.tolist()
        ]
        out.append("| " + " | ".join(cells) + " |")
    return "\n".join(out)


tier_pooled_table = (
    metrics_df.loc[metrics_df["scope"] == "feature_set_tier",
                   ["feature_set", "horizon_tier", "rmse", "mae", "wape", "bias", "n"]]
              .sort_values(["feature_set", "horizon_tier"])
    if not metrics_df.empty else pd.DataFrame()
)

report_md_lines = [
    f"# LightGBM Rolling-Origin Evaluation Report ({RUN_MODE})",
    "",
    f"Generated: {pd.Timestamp.now().isoformat(timespec='seconds')}",
    f"Eval window: {EVAL_START_DATE} .. {EVAL_END_DATE}",
    f"Origin step (days): {ORIGIN_STEP_DAYS}",
    f"Origins scored: {len(origin_schedule)}",
    f"Horizons: {HORIZONS_TO_RUN}",
    f"Feature sets: {FEATURE_SETS} (M3 is a non-operational realised-weather benchmark)",
    f"Tuning source: {project_relative(BEST_PARAMS_PATH)}",
    "",
    "## Pooled RMSE by feature_set and horizon",
    "",
    _to_pipe_table(pivot_rmse_df),
    "",
    "## Pooled feature_set x horizon_tier",
    "",
    tier_pooled_table.to_markdown(index=False, floatfmt=".3f") if not tier_pooled_table.empty else "(empty)",
    "",
    "## Notes",
    "",
    "- M3 uses realised target-day weather and is a non-operational realised-weather point-information benchmark.",
    "- Horizons h=0-h=2 use actual deterministic MEPS forecasts; h=3-h=10 use the climatology-drift synthetic-scenario emulator from notebook 04.",
    "- Tuned LightGBM parameters are taken verbatim from `outputs/lightgbm_tuning/lightgbm_optuna_best_params_full.json`; no tuning happens here.",
    "- The report is descriptive; the thesis-level conclusion is recorded separately by the authors.",
]
REPORT_MD_PATH.write_text("\n".join(report_md_lines), encoding="utf-8")
print(f"Saved: {project_relative(REPORT_MD_PATH)}")


# HTML report body.
def _df_to_html(df):
    if df is None or df.empty:
        return "<p>(empty)</p>"
    return df.to_html(index=True, border=0, escape=True)


report_html = (
    "<html><head>"
    + _table_css()
    + "</head><body>"
    + f"<h1>LightGBM Rolling-Origin Evaluation Report ({_html_escape(RUN_MODE)})</h1>"
    + f"<p>Generated: {pd.Timestamp.now().isoformat(timespec='seconds')}</p>"
    + f"<p>Eval window: {_html_escape(EVAL_START_DATE)} .. {_html_escape(EVAL_END_DATE)}. "
    + f"Origin step: {ORIGIN_STEP_DAYS} days. Origins scored: {len(origin_schedule)}. "
    + f"Horizons: {HORIZONS_TO_RUN}. Feature sets: {FEATURE_SETS}.</p>"
    + "<h2>Pooled RMSE by feature_set and horizon</h2>"
    + _df_to_html(pivot_rmse_df)
    + "<h2>Pooled feature_set x horizon_tier</h2>"
    + _df_to_html(tier_pooled_table.set_index("feature_set") if not tier_pooled_table.empty else tier_pooled_table)
    + '<p class="note">M3 uses realised target-day weather and is a non-operational realised-weather point-information benchmark. '
      'Horizons h=0-h=2 use actual deterministic MEPS forecasts; h=3-h=10 use the climatology-drift synthetic-scenario emulator from notebook 04. '
      'Tuned LightGBM parameters are taken verbatim from notebook 08; no tuning happens here. '
      'The report is descriptive; the thesis-level conclusion is recorded separately by the authors.</p>'
    + "</body></html>"
)
REPORT_HTML_PATH.write_text(report_html, encoding="utf-8")
print(f"Saved: {project_relative(REPORT_HTML_PATH)}")

# Standalone pooled-metric HTML table.
METRICS_HTML_PATH.write_text(
    "<html><head>" + _table_css() + "</head><body>"
    + f"<h1>Pooled RMSE by feature_set x horizon ({_html_escape(RUN_MODE)})</h1>"
    + _df_to_html(pivot_rmse_df)
    + "</body></html>",
    encoding="utf-8",
)
print(f"Saved: {project_relative(METRICS_HTML_PATH)}")

# Gain-summary HTML table.
if not gain_summary_df.empty:
    gain_html_pivot = gain_summary_df.pivot_table(
        index="feature_set", columns="horizon", values="pct_improvement_rmse"
    ).round(4)
    GAIN_HTML_PATH.write_text(
        "<html><head>" + _table_css() + "</head><body>"
        + f"<h1>Improvement vs M1 (RMSE pct) by feature_set x horizon ({_html_escape(RUN_MODE)})</h1>"
        + _df_to_html(gain_html_pivot)
        + "</body></html>",
        encoding="utf-8",
    )
    print(f"Saved: {project_relative(GAIN_HTML_PATH)}")


## Optional figures

If matplotlib is available, the notebook writes diagnostic horizon, comparison, horizon-tier and category-horizon figures. The CSV and Parquet files remain the canonical outputs for analysis.


In [ ]:
if _MPL_OK and not metrics_df.empty:
    fh = metrics_df.loc[metrics_df["scope"] == "feature_set_horizon"].copy()

    # Horizon-level RMSE and MAE.
    for metric, fname in [("rmse", "rmse_by_horizon_and_feature_set.png"),
                          ("mae",  "mae_by_horizon_and_feature_set.png")]:
        pivot = fh.pivot_table(index="horizon", columns="feature_set", values=metric).sort_index()
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        for fs in pivot.columns:
            ax.plot(pivot.index, pivot[fs], marker="o", label=fs)
        ax.set_xlabel("Horizon")
        ax.set_ylabel(metric.upper())
        ax.set_title(f"LightGBM rolling-origin {metric.upper()} by horizon ({RUN_MODE})")
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()
        fig.savefig(FIG_DIR / fname, dpi=140)
        plt.close(fig)
        print(f"Saved: figures{FILE_SUFFIX}/{fname}")

    # Pairwise horizon improvements.
    if not gain_summary_df.empty:
        m2_imp = gain_summary_df.loc[gain_summary_df["feature_set"] == "M2"] \
                      .groupby("horizon")["pct_improvement_rmse"].mean()
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.bar(m2_imp.index.astype(str), m2_imp.values * 100)
        ax.set_xlabel("Horizon")
        ax.set_ylabel("M2 vs M1 RMSE improvement (%)")
        ax.set_title(f"M2 vs M1 RMSE improvement by horizon ({RUN_MODE})")
        ax.grid(True, axis="y", alpha=0.3)
        fig.tight_layout()
        fig.savefig(FIG_DIR / "m2_vs_m1_improvement_by_horizon.png", dpi=140)
        plt.close(fig)
        print(f"Saved: figures{FILE_SUFFIX}/m2_vs_m1_improvement_by_horizon.png")

    if not m4_vs_m2_diagnostic_df.empty:
        m4_imp = m4_vs_m2_diagnostic_df.groupby("horizon")["m4_improvement_rmse"].mean()
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.bar(m4_imp.index.astype(str), m4_imp.values)
        ax.set_xlabel("Horizon")
        ax.set_ylabel("M4 vs M2 RMSE improvement (sales units)")
        ax.set_title(f"M4 vs M2 RMSE improvement by horizon ({RUN_MODE})")
        ax.grid(True, axis="y", alpha=0.3)
        fig.tight_layout()
        fig.savefig(FIG_DIR / "m4_vs_m2_improvement_by_horizon.png", dpi=140)
        plt.close(fig)
        print(f"Saved: figures{FILE_SUFFIX}/m4_vs_m2_improvement_by_horizon.png")

    # Pooled horizon-tier figure.
    tier = metrics_df.loc[metrics_df["scope"] == "feature_set_tier"].copy()
    if not tier.empty:
        pivot = tier.pivot_table(index="feature_set", columns="horizon_tier", values="rmse")
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        pivot.plot(kind="bar", ax=ax)
        ax.set_ylabel("Pooled RMSE")
        ax.set_title(f"Pooled RMSE by horizon tier and feature set ({RUN_MODE})")
        ax.grid(True, axis="y", alpha=0.3)
        fig.tight_layout()
        fig.savefig(FIG_DIR / "tier_summary.png", dpi=140)
        plt.close(fig)
        print(f"Saved: figures{FILE_SUFFIX}/tier_summary.png")

    # Category-horizon M2-vs-M1 improvement heatmap.
    cat_block = metrics_df.loc[metrics_df["scope"] == "feature_set_category_horizon"].copy()
    if not cat_block.empty:
        pivot_m1 = cat_block.loc[cat_block["feature_set"] == "M1"].pivot_table(
            index="Analyse_Kategori", columns="horizon", values="rmse",
        )
        pivot_m2 = cat_block.loc[cat_block["feature_set"] == "M2"].pivot_table(
            index="Analyse_Kategori", columns="horizon", values="rmse",
        )
        common_cols = sorted(set(pivot_m1.columns) & set(pivot_m2.columns))
        common_idx = sorted(set(pivot_m1.index) & set(pivot_m2.index))
        if common_cols and common_idx:
            pm1 = pivot_m1.loc[common_idx, common_cols]
            pm2 = pivot_m2.loc[common_idx, common_cols]
            improvement = (pm1 - pm2) / pm1 * 100.0
            fig, ax = plt.subplots(figsize=(8.5, 4.5))
            im = ax.imshow(improvement.values, aspect="auto", cmap="RdYlGn")
            ax.set_xticks(range(len(common_cols)))
            ax.set_xticklabels(common_cols)
            ax.set_yticks(range(len(common_idx)))
            ax.set_yticklabels(common_idx)
            ax.set_xlabel("Horizon")
            ax.set_title(f"M2 vs M1 RMSE improvement (%) by category x horizon ({RUN_MODE})")
            for i, _ in enumerate(common_idx):
                for j, _ in enumerate(common_cols):
                    v = improvement.values[i, j]
                    ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8, color="black")
            fig.colorbar(im, ax=ax, label="RMSE pct improvement vs M1")
            fig.tight_layout()
            fig.savefig(FIG_DIR / "category_weather_value_heatmap.png", dpi=140)
            plt.close(fig)
            print(f"Saved: figures{FILE_SUFFIX}/category_weather_value_heatmap.png")


## Summary

This notebook writes the rolling-origin LightGBM evaluation outputs for the configured schedule and run mode. Hyperparameters are fixed to the validation-selected LightGBM parameters from notebook 08; no tuning occurs here.

Core outputs use the configured `FILE_SUFFIX`:

- `lightgbm_rolling_origin_predictions_*.parquet`
- `lightgbm_rolling_origin_metrics_*.csv`
- `lightgbm_rolling_origin_gain_summary_*.csv`
- `lightgbm_rolling_origin_weather_value_decomposition_*.csv`
- `lightgbm_rolling_origin_m4_vs_m2_diagnostic_*.csv`
- `lightgbm_rolling_origin_horizon_tier_summary_*.csv`
- `lightgbm_rolling_origin_runtime_summary_*.csv`
- `lightgbm_rolling_origin_validation_checks_*.csv`
- `lightgbm_rolling_origin_feature_set_summary_*.csv`
- `lightgbm_rolling_origin_used_params_*.json`
- `lightgbm_rolling_origin_used_params_*.csv`
- `lightgbm_rolling_origin_metrics_*.html`
- `lightgbm_rolling_origin_gain_summary_*.html`
- `lightgbm_rolling_origin_report_*.md`
- `lightgbm_rolling_origin_report_*.html`
- Optional figures under `figures_{smoke_step5|full_step5}/`.

A smoke run should be inspected before a full run. The resulting outputs provide forecasting evidence for thesis interpretation; M3 remains a non-operational realised-weather benchmark.
